In [1]:
import pandas as pd
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
import numpy as np
import pickle
from scipy.special import softmax
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, confusion_matrix

In [1]:
%load_ext autoreload
%autoreload 1
%aimport experiments.experiment
%aimport datasets.dataset
%aimport models.model

In [2]:
from experiments.experiment import Experiment
from datasets.dataset import ReCoveryDataset
from models.model import BertBasedUncased

In [3]:
recovery_experiment = Experiment.Builder().with_name('bert base uncased').with_dataset(
    ReCoveryDataset, 'datasets/Recovery/recovery.csv').with_model(BertBasedUncased).build()

In [4]:
recovery_experiment.run()

In [2]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    probs = softmax(logits, axis=1)[:, 1]
    accuracy = accuracy_score(labels, predictions)
    recall = recall_score(labels, predictions)
    f1 = f1_score(labels, predictions)
    roc_auc = roc_auc_score(labels, probs)
    return {"accuracy": accuracy, "recall": recall, "f1": f1, "roc_auc": roc_auc}

### EUVSDISINFO

In [18]:
run = mlflow.start_run(experiment_id=bert_experiment.experiment_id)
dataset = 'euvsdisinfo'
mlflow.log_param('dataset', dataset)
df_euvsdisinfo = pd.read_csv(f'../datasets/{dataset}/{dataset}.csv')
display(df_euvsdisinfo)
mlflow.end_run()

,article,label
0,"Belarus leader, Alexander Lukashenko, met Puti...",1
1,"US secretary of state, Antony Blinken, said Ru...",1
2,"Putin: 'If attacks against Russia continue, th...",1
3,Ukrainian media has reported that the Mykolaiv...,1
4,Ukraine has announced it has killed another 48...,1
...,...,...
2134,"US, UK and Canada tried to use Ukraine as a ba...",0
2135,Ukraine continues to gather military equipment...,0
2136,Crimea reunited with Russia after the signatur...,0
2137,"Ukraine lost its statehood to the West, the sa...",0


In [5]:
train_df, test_df = train_test_split(df_euvsdisinfo, test_size=0.2, random_state=42)

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")


def tokenize_function(example):
    return tokenizer(example["article"], truncation=True, padding="max_length", max_length=256)


train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])


Parameter 'function'=<function tokenize_function at 0x00000223ABE18900> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/1711 [00:00<?, ? examples/s]

Map:   0%|          | 0/428 [00:00<?, ? examples/s]

In [10]:
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

training_args = TrainingArguments(
    output_dir="../models/results/euvsdisinfo/",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    evaluation_strategy="epoch",
    logging_steps=10,
    save_steps=100,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# Train the model
trainer.train()

# Evaluate the model without retraining
metrics = trainer.evaluate()
print(metrics)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\nikita.bardatskii\miniforge3\envs\bi-bap\Lib\site-packages\transformers\training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Recall,F1,Roc Auc
1,0.000500,0.011260,0.997664,0.995098,0.997543,1.000000
2,0.000100,0.014986,0.997664,0.995098,0.997543,1.000000
3,0.000100,0.014691,0.997664,0.995098,0.997543,1.000000


{'eval_loss': 0.014691110700368881, 'eval_accuracy': 0.9976635514018691, 'eval_recall': 0.9950980392156863, 'eval_f1': 0.9975429975429976, 'eval_roc_auc': 1.0, 'eval_runtime': 7.2612, 'eval_samples_per_second': 58.943, 'eval_steps_per_second': 7.437, 'epoch': 3.0}


In [ ]:
import os
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

mlflow.start_run()

# Get predictions
predictions = trainer.predict(test_dataset)
pred_logits = predictions.predictions
pred_labels = np.argmax(pred_logits, axis=1)

# Log metrics
metrics = compute_metrics((pred_logits, np.array(test_dataset["label"])))
for metric_name, metric_value in metrics.items():
    mlflow.log_metric(metric_name, metric_value)

# Generate confusion matrix
conf_matrix = confusion_matrix(test_dataset["label"], pred_labels)
df_cm = pd.DataFrame(conf_matrix, index=["Class 0", "Class 1"], columns=["Class 0", "Class 1"])

# Plot and save confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(df_cm, annot=True, fmt="d", cmap="Blues", xticklabels=df_cm.columns, yticklabels=df_cm.index)
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
conf_matrix_path = "confusion_matrix.png"
plt.savefig(conf_matrix_path)
plt.close()

# Log confusion matrix as an artifact
mlflow.log_artifact(conf_matrix_path)
os.remove(conf_matrix_path)

mlflow.end_run()


In [ ]:
import os
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

mlflow.start_run()

# Get predictions
predictions = trainer.predict(test_dataset)
pred_logits = predictions.predictions
pred_labels = np.argmax(pred_logits, axis=1)

# Log metrics
metrics = compute_metrics((pred_logits, np.array(test_dataset["label"])))
for metric_name, metric_value in metrics.items():
    mlflow.log_metric(metric_name, metric_value)

# Generate confusion matrix
conf_matrix = confusion_matrix(test_dataset["label"], pred_labels)
df_cm = pd.DataFrame(conf_matrix, index=["Class 0", "Class 1"], columns=["Class 0", "Class 1"])

# Plot and save confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(df_cm, annot=True, fmt="d", cmap="Blues", xticklabels=df_cm.columns, yticklabels=df_cm.index)
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
conf_matrix_path = "confusion_matrix.png"
plt.savefig(conf_matrix_path)
plt.close()

# Log confusion matrix as an artifact
mlflow.log_artifact(conf_matrix_path)
os.remove(conf_matrix_path)

mlflow.end_run()

In [13]:
with open('../models/results/euvsdisinfo/model.pkl', 'wb') as f:
    pickle.dump(model, f)

### ISOT

In [3]:
df_isot = pd.read_csv("datasets/ISOT/isot.csv")
df_isot

,article,label
0,Heaven’s Gatekeeper? Jesse Jackson Proclaims T...,0
1,Obama And Justin Trudeau Had Dinner Tuesday; ...,0
2,"U.S. willing, if asked, to facilitate talks be...",1
3,Elysee plays down opulence of Macron birthday ...,1
4,CNN Anchor Asks Van Jones To Take Back His Pra...,0
...,...,...
9995,FATHER OF STUDENT Released From N. Korean Pris...,0
9996,‘We Know Where You Live’: Trump-Loving Terror...,0
9997,New Russian ambassador to U.S. calls for resum...,1
9998,FOUR CANDIDATES FOR FBI DIRECTOR Interviewed T...,0


In [5]:
train_df, test_df = train_test_split(df_isot, test_size=0.2, random_state=42)

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")


def tokenize_function(example):
    return tokenizer(example["article"], truncation=True, padding="max_length", max_length=256)


train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])


Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [6]:
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

training_args = TrainingArguments(
    output_dir="../models/results/isot/",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    evaluation_strategy="epoch",
    logging_steps=10,
    save_steps=100,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

# Train the model
trainer.train()


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\nikita.bardatskii\miniforge3\envs\bi-bap\Lib\site-packages\transformers\training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,0.000600,0.017295
2,0.000100,0.007694
3,0.000000,0.010644


TrainOutput(global_step=3000, training_loss=0.030630216048443498, metrics={'train_runtime': 1497.7757, 'train_samples_per_second': 16.024, 'train_steps_per_second': 2.003, 'total_flos': 3157332664320000.0, 'train_loss': 0.030630216048443498, 'epoch': 3.0})

In [8]:
trainer = Trainer(
    model=model,  # Your trained model
    eval_dataset=test_dataset,  # Evaluation dataset
    compute_metrics=compute_metrics
)

# Evaluate the model without retraining
metrics = trainer.evaluate()
print(metrics)

{'eval_loss': 0.010644100606441498, 'eval_model_preparation_time': 0.003, 'eval_accuracy': 0.9985, 'eval_recall': 1.0, 'eval_f1': 0.9985815602836879, 'eval_roc_auc': 0.9999989968541345, 'eval_runtime': 32.7211, 'eval_samples_per_second': 61.123, 'eval_steps_per_second': 7.64}


In [ ]:
import pickle

with open('../models/results/isot/model.pkl', 'wb') as f:
    pickle.dump(model, f)

### Recovery

In [4]:
df_recovery = pd.read_csv("datasets/Recovery/recovery.csv")
df_recovery

,article,label
0,The Coronavirus: What Scientists Have Learned ...,1
1,Chinese Health Officials: More Die From Newly ...,1
2,Everything you need to know about the coronavi...,1
3,Novel Coronavirus Cases Confirmed To Be Spread...,0
4,Coronavirus disrupts the world: updates on the...,1
...,...,...
2024,White House concerned with coronavirus spread ...,0
2025,“There Will Be No Election If Things Keep Goin...,0
2026,Want to know who won the presidential race on ...,0
2027,Nearly half of Twitter accounts discussing cor...,0


In [5]:
train_df, test_df = train_test_split(df_recovery, test_size=0.2, random_state=42)

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")


def tokenize_function(example):
    return tokenizer(example["article"], truncation=True, padding="max_length", max_length=256)


train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])


Map:   0%|          | 0/1623 [00:00<?, ? examples/s]

Map:   0%|          | 0/406 [00:00<?, ? examples/s]

In [6]:
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

training_args = TrainingArguments(
    output_dir="../models/results/recovery/",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    evaluation_strategy="epoch",
    logging_steps=10,
    save_steps=100,
    weight_decay=0.01,
)
from transformers.integrations import MLflowCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    callbacks=[MLflowCallback()]
)

# Train the model
trainer.train()

# Evaluate the model without retraining
metrics = trainer.evaluate()
print(metrics)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\nikita.bardatskii\git\bi-bap-2025-bardanik\.venv\Lib\site-packages\transformers\training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
You are adding a <class 'transformers.integrations.integration_utils.MLflowCallback'> to the callbacks of this Trainer, but there is already one. The currentlist of callbacks is
:DefaultFlowCallback
MLflowCallback


Epoch,Training Loss,Validation Loss,Accuracy,Recall,F1,Roc Auc
1,0.372800,0.248658,0.891626,0.942529,0.917910,0.963773
2,0.232800,0.412914,0.906404,0.992337,0.931655,0.967697
3,0.079300,0.370172,0.918719,0.984674,0.939671,0.977038


{'eval_loss': 0.3701721429824829, 'eval_accuracy': 0.9187192118226601, 'eval_recall': 0.9846743295019157, 'eval_f1': 0.9396709323583181, 'eval_roc_auc': 0.9770379178226978, 'eval_runtime': 6.5935, 'eval_samples_per_second': 61.576, 'eval_steps_per_second': 7.735, 'epoch': 3.0}


In [16]:
with open('../models/results/recovery/model.pkl', 'wb') as f:
    pickle.dump(model, f)

### CZ Dataset

In [17]:
df_cz = pd.read_csv("datasets/cz_dataset/cz_dataset.csv")
df_cz

,article,label
0,MIP 24 Velká Británie hodila v Indii Ukrajinu ...,0
1,MIP 38 Brusel požaduje vyvézt evropské obilí! ...,0
2,"Senzace: Francouzský dobrovolník vyprávěl, jak...",0
3,168 hodin: Exprezident Václav Klaus se stal hv...,1
4,Ukrajina zvažuje dodávky raket dlouhého doletu...,0
...,...,...
990,Rusko v pondělí vypálilo střely za půl miliard...,1
991,Andrej Turčak chce být tváří války na Ukrajině...,1
992,"Rada bezpečnosti selhala, přiznal generální ta...",1
993,Moskva propustila přes dvě stě Ukrajinců včetn...,1


In [18]:
train_df, test_df = train_test_split(df_cz, test_size=0.2, random_state=42)

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")


def tokenize_function(example):
    return tokenizer(example["article"], truncation=True, padding="max_length", max_length=256)


train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])


Map:   0%|          | 0/796 [00:00<?, ? examples/s]

Map:   0%|          | 0/199 [00:00<?, ? examples/s]

In [19]:
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

training_args = TrainingArguments(
    output_dir="../models/results/cz_dataset/",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    evaluation_strategy="epoch",
    logging_steps=10,
    save_steps=100,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# Train the model
trainer.train()

# Evaluate the model without retraining
metrics = trainer.evaluate()
print(metrics)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\nikita.bardatskii\miniforge3\envs\bi-bap\Lib\site-packages\transformers\training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Recall,F1,Roc Auc
1,0.249400,0.383661,0.869347,0.811966,0.879630,0.972066
2,0.438200,0.242173,0.939698,0.982906,0.950413,0.984678
3,0.144500,0.224544,0.959799,0.982906,0.966387,0.986554


{'eval_loss': 0.2245437353849411, 'eval_accuracy': 0.9597989949748744, 'eval_recall': 0.9829059829059829, 'eval_f1': 0.9663865546218487, 'eval_roc_auc': 0.9865540963101938, 'eval_runtime': 4.7002, 'eval_samples_per_second': 42.339, 'eval_steps_per_second': 5.319, 'epoch': 3.0}


In [20]:
with open('../models/results/cz_dataset/model.pkl', 'wb') as f:
    pickle.dump(model, f)